# 08 — Loom: Prototype One Blocking Component

**Target:** Apache Spark 4.2.0 runtime; JDK 21+; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Does a virtual-thread prototype help the kind of wait your component actually performs?

**Why this matters:** Loom's payoff is chiefly many concurrent blocking tasks. It is not an automatic replacement for Spark's scheduler or Netty event loops.

In [ ]:
# See notebook 03 for a runnable JDK comparison. Here classify a candidate before prototyping.
profile = {
 'cpu_bound_seconds': 0.0,
 'socket_wait_seconds': 0.0,
 'monitor_wait_seconds': 0.0,
 'queue_wait_seconds': 0.0,
 'tasks_in_flight': 0,
 'resource_limit': 'document the constrained downstream resource',
}
profile

## Experiment design
1. Choose one isolated blocking JVM component.
2. Capture baseline throughput, p50/p95 latency, CPU, heap, platform/carrier threads and in-flight work.
3. Prototype virtual threads for only that component. Bound downstream access with a semaphore if needed.
4. Compare cancellation, context propagation, thread-local use and native/JNI behavior.
5. Repeat under representative load; do not infer cluster safety from a microbenchmark.

**Expected:** if waits dominate and the thread-per-blocked-task model is the bottleneck, virtual threads may improve concurrency. If CPU, downstream capacity or queueing dominates, they may not help.

**Why:** virtual threads reduce one representation/scheduling cost; they do not remove resource limits or make the work disappear.